In [1]:
import os 
os.chdir('../..')
!ls

LICENSE                demo.ipynb             food_trade
README.md              environment.yml        generate_results.ipynb


In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
from functools import reduce
import statsmodels.api as sm
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns

from food_trade.demand_supply.helper_functions import *

In [3]:
admin = gpd.read_file('../../data/admin_polygons/admin_polygons.shp')

In [4]:
crops = ['Wheat', 'Maize (corn)', 'Rye', 'Barley', 'Oats', 'Sorghum', 
         'Rice, paddy (rice milled equivalent)', 'Buckwheat', 
         'Millet', 'Quinoa', 'Canary seed', 'Fonio', 'Mixed grain', 'Triticale', 
         'Cereals n.e.c.', 'cereals_all']

for crop in crops:
    print(crop)
    df = get_totals(crop, False)

Wheat
production: 759821262.092
trade: 192360507.32999998
supply: 184782343.10737538
corr: 0.9954266084631183
corr log: 0.8594785725870379
r2: 0.9903702979262548
r2 log: 0.6687206113738968
Maize (corn)
production: 1153224941.412
trade: 183337114.91199997
supply: 177425474.78741455
corr: 0.9989786225630494
corr log: 0.8908921755643932
r2: 0.9978116939119885
r2 log: 0.7646921566530809
Rye
production: 12932937.704
trade: 1633400.406
supply: 1494443.3102729595
corr: 0.9973902270069244
corr log: 0.8909471696904947
r2: 0.9894054863151779
r2 log: 0.7632970210873024
Barley
production: 150059533.916
trade: 37852705.832
supply: 36457827.035476394
corr: 0.9969851390530942
corr log: 0.8864368934546326
r2: 0.993763583073297
r2 log: 0.7434622292410147
Oats
production: 23818690.536
trade: 4449255.546
supply: 4063872.610176231
corr: 0.9990866117546152
corr log: 0.8370316992154485
r2: 0.9976358587654882
r2 log: 0.6571959866217372
Sorghum
production: 59090591.276
trade: 7228122.155999999
supply: 7148470

### sensitivity analysis

In [5]:
for crop in ['wheat', 'rice', 'maize', 'other_cereals', 'cereals_all', 'combined_crops']:
    
    print(crop)
    
    df_beta_1_5 = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.5.parquet.gzip')
    df_beta_1_2 = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.2.parquet.gzip')
       
    traded_base = df_beta_1_5[df_beta_1_5['is_self'] == False]
    traded_test = df_beta_1_2[df_beta_1_2['is_self'] == False]
    
    merged_df, corr, cpf = calculate_sensitivity_metrics(
        df_base=traded_base, 
        df_test=traded_test, 
        description=f"{crop} flows: beta=1.5 vs beta=1.2"
    )

    df_beta_1_5 = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.5.parquet.gzip')
    df_beta_1_8 = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.8.parquet.gzip')
       
    traded_base = df_beta_1_5[df_beta_1_5['is_self'] == False]
    traded_test = df_beta_1_8[df_beta_1_8['is_self'] == False]
    
    merged_df, corr, cpf = calculate_sensitivity_metrics(
        df_base=traded_base, 
        df_test=traded_test, 
        description=f"{crop} flows: beta=1.5 vs beta=1.8"
    )

    df_beta_1_8 = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.8.parquet.gzip')
    df_beta_1_2 = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.2.parquet.gzip')
       
    traded_base = df_beta_1_8[df_beta_1_8['is_self'] == False]
    traded_test = df_beta_1_2[df_beta_1_2['is_self'] == False]
    
    merged_df, corr, cpf = calculate_sensitivity_metrics(
        df_base=traded_base, 
        df_test=traded_test, 
        description=f"{crop} flows: beta=1.8 vs beta=1.2"
    )

wheat

SENSITIVITY ANALYSIS: wheat flows: beta=1.5 vs beta=1.2
  Total Links Base: 580,715
  Total Links Test: 581,995
  Union of Links:   583,259

  --- Metrics ---
  Spearman Rank Correlation: 0.9997 (p-val: 0.00e+00)
  Common Part of Flows (CPF): 0.9804
  Top 5% Heavy Hitter Overlap: 98.6%
  R2: 0.9979582604690723%

SENSITIVITY ANALYSIS: wheat flows: beta=1.5 vs beta=1.8
  Total Links Base: 580,715
  Total Links Test: 578,937
  Union of Links:   581,858

  --- Metrics ---
  Spearman Rank Correlation: 0.9997 (p-val: 0.00e+00)
  Common Part of Flows (CPF): 0.9805
  Top 5% Heavy Hitter Overlap: 98.7%
  R2: 0.9979420106279538%

SENSITIVITY ANALYSIS: wheat flows: beta=1.8 vs beta=1.2
  Total Links Base: 578,937
  Total Links Test: 581,995
  Union of Links:   584,401

  --- Metrics ---
  Spearman Rank Correlation: 0.9987 (p-val: 0.00e+00)
  Common Part of Flows (CPF): 0.9610
  Top 5% Heavy Hitter Overlap: 97.3%
  R2: 0.9919809780592791%
rice

SENSITIVITY ANALYSIS: rice flows: beta=1.5 vs 

### validation

#### US FAF data

In [6]:
net_faf = process_faf_net_flows(admin)

Extracted 12,253 gross flow records for grains.
  Extracted 871 active net subnational links.


In [7]:
df_combined = pd.read_parquet('../../data/outputs/flows_combined_crops_1.5.parquet.gzip')

# Filter model's output for US domestic trade only
df_us_model = df_combined[
    (df_combined['from_iso3'] == 'USA') & 
    (df_combined['to_iso3'] == 'USA') & 
    (df_combined['is_self'] == False)
].copy()

print(f"Extracted {len(df_us_model):,} US domestic links from the model.")

# Merge the FAF empirical data with your theoretical model data
merged_df = pd.merge(
    net_faf[['from_id', 'to_id', 'flow']].rename(columns={'flow': 'flow_faf'}), 
    df_us_model[['from_id', 'to_id', 'flow']].rename(columns={'flow': 'flow_model'}), 
    on=['from_id', 'to_id'],
    how='outer'
)
merged_df['flow_faf'] = merged_df['flow_faf'].fillna(0)
merged_df['flow_model'] = merged_df['flow_model'].fillna(0) # the extra links in model could be due to cereal disaggregation 
                                                            # we are calculating the net in faf without caring for cereal type

Extracted 1,082 US domestic links from the model.


In [8]:
print(f"\n{'='*50}")
print(f"NODAL VALIDATION: Origin & Destination Balances")
print(f"{'='*50}")

# Aggregate Total OUTFLOWS by State (Who are the net exporters?)
# Group by the origin ID
outflows_faf = merged_df.groupby('from_id')['flow_faf'].sum().reset_index()
outflows_model = merged_df.groupby('from_id')['flow_model'].sum().reset_index()

out_merged = pd.merge(outflows_faf, outflows_model, on='from_id', how='outer').fillna(0)
out_spearman, out_p = spearmanr(out_merged['flow_faf'], out_merged['flow_model'])

# Aggregate Total INFLOWS by State (Who are the net importers?)
# Group by the destination ID
inflows_faf = merged_df.groupby('to_id')['flow_faf'].sum().reset_index()
inflows_model = merged_df.groupby('to_id')['flow_model'].sum().reset_index()
    
in_merged = pd.merge(inflows_faf, inflows_model, on='to_id', how='outer').fillna(0)
in_spearman, in_p = spearmanr(in_merged['flow_faf'], in_merged['flow_model'])

print(f"Outflow Rank Correlation:  {out_spearman:.4f} (p-val: {out_p:.2e})")
print(f"Outflow R2:  {r2_score(out_merged['flow_faf'], out_merged['flow_model'])}")
print(f"Inflow Rank Correlation:   {in_spearman:.4f} (p-val: {in_p:.2e})")
print(f"Inflow R2:  {r2_score(in_merged['flow_faf'], in_merged['flow_model'])}")



NODAL VALIDATION: Origin & Destination Balances
Outflow Rank Correlation:  0.6962 (p-val: 1.41e-08)
Outflow R2:  0.452105011655653
Inflow Rank Correlation:   0.2557 (p-val: 7.02e-02)
Inflow R2:  0.10108957051874035


In [9]:
print('top 5 destination states in FAF')
print(inflows_faf.sort_values('flow_faf', ascending=False).head().merge(admin[['ID', 'admin_name']].rename(columns={'ID': 'to_id'})))

print('top 5 destination states in your Model')
print(inflows_model.sort_values('flow_model', ascending=False).head().merge(admin[['ID', 'admin_name']].rename(columns={'ID': 'to_id'})))

top 5 destination states in FAF
      to_id      flow_faf  admin_name
0  USA.19_1  4.784161e+07   Louisiana
1  USA.44_1  3.188805e+07       Texas
2  USA.48_1  1.600691e+07  Washington
3  USA.11_1  1.403013e+07     Georgia
4  USA.15_1  1.138256e+07     Indiana
top 5 destination states in your Model
      to_id    flow_model      admin_name
0  USA.44_1  2.176352e+07           Texas
1   USA.5_1  1.806211e+07      California
2  USA.34_1  1.326969e+07  North Carolina
3  USA.39_1  1.322193e+07    Pennsylvania
4  USA.11_1  1.265966e+07         Georgia


#### India data from harris et al

In [48]:
harris_dom = process_harris_data(admin)

In [22]:
def nodal_validation(merged_df, crop_name):
    print(f"\n{'='*50}")
    print(f"INDIA NODAL VALIDATION: {crop_name.upper()}")
    print(f"{'='*50}")

    # 1. Outflows (Net Exporters)
    outflows_harris = merged_df.groupby('from_id')['flow_harris'].sum().reset_index()
    outflows_model = merged_df.groupby('from_id')['flow_model'].sum().reset_index()
    out_merged = pd.merge(outflows_harris, outflows_model, on='from_id', how='outer').fillna(0)
    out_spearman, out_p = spearmanr(out_merged['flow_harris'], out_merged['flow_model'])

    # 2. Inflows (Net Importers)
    inflows_harris = merged_df.groupby('to_id')['flow_harris'].sum().reset_index()
    inflows_model = merged_df.groupby('to_id')['flow_model'].sum().reset_index()
    in_merged = pd.merge(inflows_harris, inflows_model, on='to_id', how='outer').fillna(0)
    in_spearman, in_p = spearmanr(in_merged['flow_harris'], in_merged['flow_model'])

    print(f"Outflow Rank Correlation:  {out_spearman:.4f} (p-val: {out_p:.2e})")
    print(f"Outflow R2:  {r2_score(out_merged['flow_harris'], out_merged['flow_model'])}")
    print(f"Inflow Rank Correlation:   {in_spearman:.4f} (p-val: {in_p:.2e})")
    print(f"Inflow R2:  {r2_score(in_merged['flow_harris'], in_merged['flow_model'])}")
    
    return out_merged, in_merged
    
def link_validation(merged_df, crop_name):
    """Calculates Spearman rank and Common Part of Flows for specific routes."""
    # Only look at the routes, ignore the nodes for a second
    spearman_corr, p_value = spearmanr(merged_df['flow_harris'], merged_df['flow_model'])
    
    # Calculate Common Part of Flows (CPF)
    min_flows = np.minimum(merged_df['flow_harris'], merged_df['flow_model'])
    total_harris = merged_df['flow_harris'].sum()
    total_model = merged_df['flow_model'].sum()
    
    # Avoid division by zero if a crop has 0 flow
    if (total_harris + total_model) == 0:
        cpf = 0
    else:
        cpf = (2 * min_flows.sum()) / (total_harris + total_model)

    print(f"  --- LINK VALIDATION ({crop_name.upper()}) ---")
    print(f"  Link Rank Correlation:      {spearman_corr:.4f} (p-val: {p_value:.2e})")
    print(f"  Common Part of Flows (CPF): {cpf:.4f}")
    print(f"  R2: {r2_score(merged_df['flow_harris'], merged_df['flow_model'])}\n")
    
    return spearman_corr, cpf

for crop in ['wheat', 'rice', 'maize', 'other_cereals', 'combined_crops']:
    
    model_crop = pd.read_parquet(f'../../data/outputs/flows_{crop}_1.5.parquet.gzip')
    model_ind = model_crop[
        (model_crop['from_iso3'] == 'IND') & 
        (model_crop['to_iso3'] == 'IND') & 
        (model_crop['is_self'] == False)
    ]
    
    merged_crop = pd.merge(
        harris_dom[['from_id', 'to_id', f'supply_{crop}']].rename(columns={f'supply_{crop}': 'flow_harris'}), 
        model_ind[['from_id', 'to_id', 'flow']].rename(columns={'flow': 'flow_model'}), 
        on=['from_id', 'to_id'], 
        how='outer'
    ).fillna(0)
    
    # Call Nodal Validation
    _,_ = nodal_validation(merged_crop, crop)
    
    # Call the new Link Validation
    _,_ = link_validation(merged_crop, crop)


INDIA NODAL VALIDATION: WHEAT
Outflow Rank Correlation:  0.7353 (p-val: 4.87e-07)
Outflow R2:  0.5063062085165131
Inflow Rank Correlation:   0.5024 (p-val: 1.79e-03)
Inflow R2:  0.25445421731391826
  --- LINK VALIDATION (WHEAT) ---
  Link Rank Correlation:      0.6592 (p-val: 6.67e-150)
  Common Part of Flows (CPF): 0.4273
  R2: 0.04793339304454569


INDIA NODAL VALIDATION: RICE
Outflow Rank Correlation:  0.5247 (p-val: 1.02e-03)
Outflow R2:  0.6630659661880915
Inflow Rank Correlation:   0.0636 (p-val: 7.16e-01)
Inflow R2:  -0.43141646064820116
  --- LINK VALIDATION (RICE) ---
  Link Rank Correlation:      0.2048 (p-val: 5.79e-13)
  Common Part of Flows (CPF): 0.2543
  R2: -0.23087922427154983


INDIA NODAL VALIDATION: MAIZE
Outflow Rank Correlation:  0.1494 (p-val: 3.84e-01)
Outflow R2:  -1.5617945805439701
Inflow Rank Correlation:   0.0301 (p-val: 8.64e-01)
Inflow R2:  -0.9032687551044734
  --- LINK VALIDATION (MAIZE) ---
  Link Rank Correlation:      0.0290 (p-val: 3.11e-01)
  Comm